In [2]:
# Import required packages
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.metrics import r2_score, mean_squared_error
import matplotlib.pyplot as plt
import seaborn as sns

c:\Users\skazempour\AppData\Local\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
# Directories
PROJECT_DIR = r"C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/"

CODE_DIR = os.path.join(PROJECT_DIR, "Code")
DATA_DIR = os.path.join(PROJECT_DIR, "Data")
FIGURES_DIR = os.path.join(PROJECT_DIR, "Figures")
TABLES_DIR = os.path.join(PROJECT_DIR, "Tables")

# File names
INPUT_DATA = os.path.join(DATA_DIR, "merged_master.pkl")

In [4]:
data = pd.read_pickle(INPUT_DATA)

In [5]:
# Prepare features and target
# Target variable
TARGET = 'ar_FF5_1'

# Feature engineering
# data['log_n_tweets'] = np.log10(1 + data['n_tweets'])

# Select features
FEATURES = ['net_sentiment', 'log_volume']

# Remove missing values
model_data = data[[TARGET] + FEATURES].dropna()

print(f"Sample size: {len(model_data):,}")
print(f"Target: {TARGET}")
print(f"Features: {FEATURES}")

Sample size: 16,755,975
Target: ar_FF5_1
Features: ['net_sentiment', 'log_volume']


# In-Sample linear regression

In [6]:
# Prepare X and y
X = model_data[FEATURES]
y = model_data[TARGET]

# Fit linear regression model (in-sample)
lr_model = LinearRegression()
lr_model.fit(X, y)

# Make predictions
y_pred = lr_model.predict(X)

# Evaluate performance
r2 = r2_score(y, y_pred)
mse = mean_squared_error(y, y_pred)
rmse = np.sqrt(mse)

print("In-Sample Linear Regression Results")
print("=" * 50)
print(f"R-squared: {r2:.6f}")
print(f"RMSE: {rmse:.6f}")
print(f"MSE: {mse:.6f}")
print("\nCoefficients:")
for feature, coef in zip(FEATURES, lr_model.coef_):
    print(f"  {feature}: {coef:.6f}")
print(f"  Intercept: {lr_model.intercept_:.6f}")

In-Sample Linear Regression Results
R-squared: -0.001372
RMSE: 0.042364
MSE: 0.001795

Coefficients:
  net_sentiment: -0.001684
  log_volume: 0.001810
  Intercept: -0.000283


# OOS predictions

In [7]:
# Out-of-sample predictions with multiple window types
# Parameters
TRAIN_END_DATE = '2011-12-31'
WINDOW_252 = 252  # One trading year
WINDOW_21 = 21    # One trading month

# Ensure date column is datetime
model_data['date'] = pd.to_datetime(data.loc[model_data.index, 'date'])

# Sort by date
model_data = model_data.sort_values('date')

# Get unique dates
unique_dates = model_data['date'].unique()
unique_dates = pd.DatetimeIndex(unique_dates).sort_values()

# Find the starting prediction date (first date after training window)
train_end = pd.to_datetime(TRAIN_END_DATE)
oos_dates = unique_dates[unique_dates > train_end]

print(f"Training window: {unique_dates[0].strftime('%Y-%m-%d')} to {TRAIN_END_DATE}")
print(f"OOS prediction period: {oos_dates[0].strftime('%Y-%m-%d')} to {oos_dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of OOS dates: {len(oos_dates):,}")

# Initialize storage for predictions
predictions_expanding = []
predictions_rolling_252 = []
predictions_rolling_21 = []

# Loop through OOS dates
for i, pred_date in enumerate(oos_dates):
    # Test data: observations on pred_date
    test_mask = model_data['date'] == pred_date
    X_test = model_data.loc[test_mask, FEATURES]
    
    if len(X_test) == 0:
        continue
    
    test_indices = model_data.index[test_mask]
    
    # 1. Expanding window: all data up to pred_date
    train_mask_exp = model_data['date'] < pred_date
    X_train_exp = model_data.loc[train_mask_exp, FEATURES]
    y_train_exp = model_data.loc[train_mask_exp, TARGET]
    
    if len(X_train_exp) > 0:
        lr_exp = LinearRegression()
        lr_exp.fit(X_train_exp, y_train_exp)
        y_pred_exp = lr_exp.predict(X_test)
        
        for idx, pred in zip(test_indices, y_pred_exp):
            predictions_expanding.append({
                'date': pred_date,
                'index': idx,
                'pred_expanding': pred
            })
    
    # 2. Rolling 252-day window
    date_idx = unique_dates.get_loc(pred_date)
    if date_idx >= WINDOW_252:
        start_date_252 = unique_dates[date_idx - WINDOW_252]
        train_mask_252 = (model_data['date'] >= start_date_252) & (model_data['date'] < pred_date)
        X_train_252 = model_data.loc[train_mask_252, FEATURES]
        y_train_252 = model_data.loc[train_mask_252, TARGET]
        
        if len(X_train_252) > 0:
            lr_252 = LinearRegression()
            lr_252.fit(X_train_252, y_train_252)
            y_pred_252 = lr_252.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_252):
                predictions_rolling_252.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_252': pred
                })
    
    # 3. Rolling 21-day window
    if date_idx >= WINDOW_21:
        start_date_21 = unique_dates[date_idx - WINDOW_21]
        train_mask_21 = (model_data['date'] >= start_date_21) & (model_data['date'] < pred_date)
        X_train_21 = model_data.loc[train_mask_21, FEATURES]
        y_train_21 = model_data.loc[train_mask_21, TARGET]
        
        if len(X_train_21) > 0:
            lr_21 = LinearRegression()
            lr_21.fit(X_train_21, y_train_21)
            y_pred_21 = lr_21.predict(X_test)
            
            for idx, pred in zip(test_indices, y_pred_21):
                predictions_rolling_21.append({
                    'date': pred_date,
                    'index': idx,
                    'pred_rolling_21': pred
                })
    
    # Progress update
    if (i + 1) % 100 == 0:
        print(f"Processed {i + 1}/{len(oos_dates)} dates ({100 * (i + 1) / len(oos_dates):.1f}%)")

print(f"\nCompleted.")
print(f"Expanding window predictions: {len(predictions_expanding):,}")
print(f"Rolling 252-day predictions: {len(predictions_rolling_252):,}")
print(f"Rolling 21-day predictions: {len(predictions_rolling_21):,}")

Training window: 2010-01-04 to 2011-12-31
OOS prediction period: 2012-01-03 to 2024-12-31
Number of OOS dates: 3,270
Processed 100/3270 dates (3.1%)
Processed 200/3270 dates (6.1%)
Processed 300/3270 dates (9.2%)
Processed 400/3270 dates (12.2%)
Processed 500/3270 dates (15.3%)
Processed 600/3270 dates (18.3%)
Processed 700/3270 dates (21.4%)
Processed 800/3270 dates (24.5%)
Processed 900/3270 dates (27.5%)
Processed 1000/3270 dates (30.6%)
Processed 1100/3270 dates (33.6%)
Processed 1200/3270 dates (36.7%)
Processed 1300/3270 dates (39.8%)
Processed 1400/3270 dates (42.8%)
Processed 1500/3270 dates (45.9%)
Processed 1600/3270 dates (48.9%)
Processed 1700/3270 dates (52.0%)
Processed 1800/3270 dates (55.0%)
Processed 1900/3270 dates (58.1%)
Processed 2000/3270 dates (61.2%)
Processed 2100/3270 dates (64.2%)
Processed 2200/3270 dates (67.3%)
Processed 2300/3270 dates (70.3%)
Processed 2400/3270 dates (73.4%)
Processed 2500/3270 dates (76.5%)
Processed 2600/3270 dates (79.5%)
Processed 2

In [10]:
# Convert predictions to DataFrames and merge
df_expanding = pd.DataFrame(predictions_expanding)
df_rolling_252 = pd.DataFrame(predictions_rolling_252)
df_rolling_21 = pd.DataFrame(predictions_rolling_21)

# Merge all predictions together
predictions_df = df_expanding.merge(
    df_rolling_252,
    on=['date', 'index'],
    how='outer'
).merge(
    df_rolling_21,
    on=['date', 'index'],
    how='outer'
)

# Add symbol information
predictions_df = predictions_df.merge(
    data[['permno', 'ticker']].reset_index(),
    left_on='index',
    right_on='index',
    how='left'
)

# Reorder columns
predictions_df = predictions_df[['date', 'permno', 'ticker', 'index', 'pred_expanding', 'pred_rolling_252', 'pred_rolling_21']]

# Sort by date and symbol
predictions_df = predictions_df.sort_values(['date', 'ticker']).reset_index(drop=True)

print("Predictions Summary")
print("=" * 50)
print(f"Total observations: {len(predictions_df):,}")
print(f"\nNon-null predictions by window type:")
print(f"  Expanding: {predictions_df['pred_expanding'].notna().sum():,}")
print(f"  Rolling 252-day: {predictions_df['pred_rolling_252'].notna().sum():,}")
print(f"  Rolling 21-day: {predictions_df['pred_rolling_21'].notna().sum():,}")
print(f"\nFirst 10 predictions:")
print(predictions_df.head(10))

Predictions Summary
Total observations: 14,557,219

Non-null predictions by window type:
  Expanding: 14,557,219
  Rolling 252-day: 14,557,219
  Rolling 21-day: 14,557,219

First 10 predictions:
        date  permno ticker    index  pred_expanding  pred_rolling_252  \
0 2012-01-03   87432      A  2879304        0.000162         -0.000085   
1 2012-01-03   24643     AA  2377768        0.000162         -0.000085   
2 2012-01-03   12479    AAC  2266267        0.000162         -0.000085   
3 2012-01-03   90020   AACC  2994307        0.000162         -0.000085   
4 2012-01-03   15580   AAME  2343833        0.000162         -0.000085   
5 2012-01-03   10517    AAN  2211052        0.000162         -0.000085   
6 2012-01-03   76868   AAON  2576068        0.000162         -0.000085   
7 2012-01-03   89217    AAP  2947508        0.000162         -0.000085   
8 2012-01-03   14593   AAPL  2339391       -0.000402         -0.000510   
9 2012-01-03   90854   AATI  3056541        0.000162         -0.0

In [11]:
# Save predictions to Data directory
OUTPUT_DATA_DIR = os.path.join(PROJECT_DIR, "Data")
os.makedirs(OUTPUT_DATA_DIR, exist_ok=True)

OUTPUT_FILE = os.path.join(OUTPUT_DATA_DIR, "predictions_linear_regression.pkl")
predictions_df.to_pickle(OUTPUT_FILE)

print(f"Predictions saved to: {OUTPUT_FILE}")
print(f"File size: {os.path.getsize(OUTPUT_FILE) / (1024**2):.2f} MB")
print(f"Shape: {predictions_df.shape}")

Predictions saved to: C:/Users/skazempour/Dropbox/Projects/42 - Machine learning from the crowd/Data\predictions_linear_regression.pkl
File size: 758.14 MB
Shape: (14557219, 7)
